**Fine-Tuning the DistilBERT based Cased Model: Round 1**

Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Install Package

In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops wandb

Import Library

In [ ]:
from datasets import load_dataset, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments
import pandas as pd
import torch
import os

Dataset

In [ ]:
# Train Dataset
train_dataset = load_dataset('fancyzhx/ag_news', split="train")
train_dataset = pd.DataFrame(train_dataset)
train_dataset = Dataset.from_pandas(train_dataset)

# Test Dataset
test_dataset = load_dataset('fancyzhx/ag_news', split="test")
test_dataset = pd.DataFrame(test_dataset)
test_dataset = Dataset.from_pandas(test_dataset)

Labels

In [ ]:
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

Model (Before Fine-Tune)

In [ ]:
model_name = 'distilbert-base-cased'

# quantization_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = DistilBertForSequenceClassification.from_pretrained(model_name,quantization_config=bnb_config, num_labels=4)
based_model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=4)
model.config.use_cache = False

In [ ]:
print(model)

In [ ]:
print(based_model)

Tokenization

In [ ]:
# Preprocessing
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# Tokenize the dataset
def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
    tokenized['labels'] = examples['label']  # Ensure labels are added as 'labels'
    return tokenized

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_train_dataset.set_format('torch')

# Preprocess the test dataset
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)
tokenized_test_dataset.set_format('torch')

In [ ]:
# คำนวณจำนวนพารามิเตอร์ทั้งหมดในโมเดล
num_params = sum(p.numel() for p in model.parameters())

# คำนวณขนาดประมาณของโมเดลในหน่วย MB
# โดยพิจารณาแต่ละพารามิเตอร์ใช้ 4 bytes (float32)
model_size_approx = num_params * 4 / (1024 * 1024)  # ขนาดใน MB
print(f"Approximate model size before fine-tuning: {model_size_approx:.2f} MB")

In [ ]:
# คำนวณจำนวนพารามิเตอร์ทั้งหมดในโมเดล
num_params = sum(p.numel() for p in based_model.parameters())

# คำนวณขนาดประมาณของโมเดลในหน่วย MB
# โดยพิจารณาแต่ละพารามิเตอร์ใช้ 4 bytes (float32)
model_size_approx = num_params * 4 / (1024 * 1024)  # ขนาดใน MB
print(f"Approximate model size before fine-tuning: {model_size_approx:.2f} MB")

Set training data batch size

In [ ]:
per_device_train_batch_size = 16
gradient_accumulation_steps = 2
total_train_data_size = len(train_dataset)  # Total training data size
batch_size = per_device_train_batch_size * 1  # Assuming 1 GPU
steps_per_epoch = int(total_train_data_size / (batch_size * gradient_accumulation_steps))
desired_epochs = 1  # Example of desired number of epochs

# Calculate max_steps for TrainingArguments
max_steps_calculated = steps_per_epoch * desired_epochs

Accuracy before fine-tuning

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [ ]:
# Set model to evaluation mode
model.eval()

# Store results
predictions = []
true_labels = []

# Model testing
for batch in DataLoader(tokenized_test_dataset, batch_size=per_device_train_batch_size):
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1).tolist()

    predictions.extend(predicted_labels)
    true_labels.extend(batch['labels'].tolist())

# Calculate accuracy
accuracy_before_finetune = accuracy_score(true_labels, predictions)
print(f"Accuracy before fine-tuning: {accuracy_before_finetune:.4f}")

# Generate classification report
report = classification_report(true_labels, predictions, target_names=list(labels.values()))
print(report)

Accuracy before fine-tuning: 0.2509
              precision    recall  f1-score   support

       World       0.20      0.01      0.03      1900
      Sports       0.25      0.99      0.40      1900
    Business       0.00      0.00      0.00      1900
    Sci/Tech       0.00      0.00      0.00      1900

    accuracy                           0.25      7600
   macro avg       0.11      0.25      0.11      7600
weighted avg       0.11      0.25      0.11      7600



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fine-Tuning

In [ ]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="SEQ_CLASS_LM", #Fix
    target_modules=[
        "attention.q_lin",
        "attention.k_lin",
        "attention.v_lin"    ]
)

In [ ]:
output_dir = "/content/drive/MyDrive/llm_project_cased/"
os.makedirs(output_dir, exist_ok=True)
gradient_accumulation_steps = gradient_accumulation_steps
optim = "paged_adamw_32bit"
save_steps = 1000
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 100
warmup_ratio = 0.03
lr_scheduler_type = "cosine"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps_calculated,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
)

Trainer

In [ ]:
from trl import SFTTrainer
from transformers import DefaultDataCollator

max_seq_length = 512

def formatting_func(example):
    return example["text"]

data_collator = DefaultDataCollator()

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    peft_config=peft_config,
    dataset_text_field="input_ids",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    data_collator=data_collator,
)

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

In [ ]:
trainer.train()

In [ ]:
print(model)

Accuracy after fine-tuning

In [ ]:
# Set model to evaluation mode
model.eval()

# Store results
predictions = []
true_labels = []

# Model testing
for batch in DataLoader(tokenized_test_dataset, batch_size=per_device_train_batch_size):
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predicted_labels = torch.argmax(logits, dim=-1).tolist()

    predictions.extend(predicted_labels)
    true_labels.extend(batch['labels'].tolist())

# Calculate accuracy
accuracy_after_finetune = accuracy_score(true_labels, predictions)
print(f"Accuracy after fine-tuning: {accuracy_after_finetune:.4f}")

# Generate classification report
report = classification_report(true_labels, predictions, target_names=list(labels.values()))
print(report)

Accuracy after fine-tuning: 0.9109
              precision    recall  f1-score   support

       World       0.93      0.89      0.91      1900
      Sports       0.96      0.98      0.97      1900
    Business       0.87      0.88      0.87      1900
    Sci/Tech       0.88      0.89      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



Save Model

In [ ]:
# Define path for saving the model
save_directory = "/content/drive/MyDrive/llm_project_cased/full_model"

# Create folder if it doesn't already exist
os.makedirs(save_directory, exist_ok=True)

# 1. Convert PEFT model to full model
model = trainer.model.merge_and_unload()

# 2. Save the full model
model.save_pretrained(save_directory)

# 3. Save the tokenizer
tokenizer.save_pretrained(save_directory)

# 4. Save the config
model.config.save_pretrained(save_directory)

print(f"Full model and tokenizer have been saved to {save_directory}")

# Test loading the model back
try:
    loaded_model = DistilBertForSequenceClassification.from_pretrained(save_directory)
    loaded_tokenizer = DistilBertTokenizer.from_pretrained(save_directory)
    print("Model loaded successfully!")

    # Test prediction
    test_text = "Your test text here"
    inputs = loaded_tokenizer(test_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = loaded_model(**inputs)
    print("Prediction test successful!")

except Exception as e:
    print(f"An error occurred: {str(e)}")